# 🚀 NINE-1 — Pré-Treinamento no Google Colab

Passos:
1. Instala dependencias minimas (PyTorch ja vem no Colab)
2. Clona o repositorio do NINE-1
3. Gera corpus seed (ou baixa um)
4. Treina o BPE tokenizer
5. Roda pre-treinamento (algumas horas na T4 gratuita)
6. (Opcional) Fine-tune LoRA em dados instrucionais
7. Exporta checkpoint para usar na CLI local

**Tempo orcado:** ~2-4h na T4 gratuita para um modelo base (~10-50M).

In [ ]:
# (1) Setup
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
!pip install -q numpy

In [ ]:
# (2) Clone o repositorio nine-1 (substitua a URL pela sua)
import os
if not os.path.exists('nine-1'):
    # aqui voce faria: !git clone https://github.com/SEU_USUARIO/nine-1.git
    # Para executar sem git: vamos usar upload manual ou copia
    pass
print('OK')

In [ ]:
# (3) Gera corpus seed e prepara dados
%cd nine-1
!python scripts/download_seed.py
!python -m nine.prep_data --paths nine/data/seed --train_bpe --vocab 4096 --max_chars 5000000
import os
print('Arquivos:', os.listdir('nine/data'))

In [ ]:
# (4) Pre-treinamento do NINE-1 base
# Modelo ~30M params (10 layers, 8 heads, 512 embd, ctx=512)
!python -m nine.train \
    --data nine/data/corpus.bin \
    --out nine/data/nine1-base.pt \
    --vocab 4096 \
    --block_size 512 \
    --n_layer 10 \
    --n_head 8 \
    --n_embd 512 \
    --batch_size 16 \
    --max_iters 2000 \
    --lr 3e-4

In [ ]:
# (5) Fine-tuning LoRA (em dataset de instrucoes)
# Primeiro, gere um dataset pequeno de instrucoes usando o proprio NINE-1 ou copie um
INSTRUCT_SAMPLE = '''{"instruction":"escreva uma funcao soma em python","output":"def soma(a, b):\\n    return a + b"}
{"instruction":"como fazer um loop for","output":"for i in range(10):\\n    print(i)"}
{"instruction":"escreva fibonacci iterativo","output":"def fib(n):\\n    a, b = 0, 1\\n    for _ in range(n):\\n        a, b = b, a+b\\n    return a"}
'''
import os
os.makedirs('nine/data', exist_ok=True)
with open('nine/data/instruct.jsonl', 'w') as f:
    f.write(INSTRUCT_SAMPLE)
print('Dataset salvo')

In [ ]:
# (6) Fine-tune com LoRA
!python -m nine.finetune \
    --base nine/data/nine1-base.pt \
    --data nine/data/instruct.jsonl \
    --out nine/data/nine1-instruct.pt \
    --lora_r 8 \
    --lora_alpha 16 \
    --max_iters 200 \
    --lr 2e-4

In [ ]:
# (7) Testa o modelo gerando codigo
!python -m nine.cli "def soma(a, b):" --ckpt nine/data/nine1-base.pt --tokens 80 --temp 0.7

In [ ]:
# (8) Exporta checkpoint para baixar
from google.colab import files
files.download('nine/data/nine1-base.pt')
# files.download('nine/data/nine1-instruct.pt')
# Salve esses .pt no diretorio nine/data do seu projeto local Termux